# Student Academic Performance Prediction Using Machine Learning

This notebook implements a complete classification project using the Student Performance dataset. The goal is to predict whether a student will pass or fail based on academic, demographic, and social features.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance

## 2. Load the dataset

In [ ]:
df = pd.read_csv("student_data.csv")
print("Dataset shape:", df.shape)
df.head()

## 3. Basic data exploration

In [ ]:
print(df.info())
print("Missing values:", df.isnull().sum().sum())
df.describe()

## 4. Create the classification target

The original final grade column `G3` is numeric from 0 to 20. To match the project proposal, it is converted into a binary classification target:

- `1 = Pass` if `G3 >= 10`
- `0 = Fail` if `G3 < 10`

In [ ]:
df["Pass_Fail"] = (df["G3"] >= 10).astype(int)
print(df["Pass_Fail"].value_counts())

df["Pass_Fail"].map({0: "Fail", 1: "Pass"}).value_counts().reindex(["Fail", "Pass"]).plot(kind="bar", figsize=(6,4))
plt.title("Target Distribution: Pass vs Fail")
plt.xlabel("Class")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

## 5. Prepare features and preprocessing

Categorical variables are encoded using One-Hot Encoding. Numerical variables are scaled using StandardScaler. The dataset is split into 80% training and 20% testing.

In [ ]:
feature_cols = [c for c in df.columns if c not in ["G3", "Pass_Fail"]]
X = df[feature_cols]
y = df["Pass_Fail"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 6. Train and tune models

Four classification models are trained and compared: Logistic Regression, Decision Tree, Random Forest, and Support Vector Machine. GridSearchCV is used to tune the main hyperparameters.

In [ ]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=42), {"clf__C": [0.1, 1, 10]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=42), {"clf__max_depth": [3, 5, None], "clf__min_samples_leaf": [1, 5]}),
    "Random Forest": (RandomForestClassifier(random_state=42, n_jobs=1), {"clf__n_estimators": [50], "clf__max_depth": [5, None], "clf__min_samples_leaf": [1, 5]}),
    "SVM": (SVC(random_state=42), {"clf__C": [0.1, 1, 10], "clf__kernel": ["linear"]})
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
results = []
best_estimators = {}
confusion_matrices = {}

for name, (estimator, params) in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("clf", estimator)])
    grid = GridSearchCV(pipe, params, cv=cv, scoring="f1", n_jobs=1)
    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)

    results.append({
        "Model": name,
        "Best Parameters": str(grid.best_params_),
        "CV F1": round(grid.best_score_, 4),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1-score": round(f1_score(y_test, y_pred), 4)
    })
    best_estimators[name] = grid.best_estimator_
    confusion_matrices[name] = confusion_matrix(y_test, y_pred)

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False).reset_index(drop=True)
results_df

## 7. Visualize model performance

In [ ]:
results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score"]].plot(kind="bar", figsize=(9,5))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 8. Confusion matrix for the best model

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = best_estimators[best_model_name]
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
print("Best model:", best_model_name)
print(cm)
print(classification_report(y_test, y_pred_best, target_names=["Fail", "Pass"]))

fig, ax = plt.subplots(figsize=(4,4))
ax.imshow(cm)
ax.set_title(f"Confusion Matrix - {best_model_name}")
ax.set_xticks([0,1], labels=["Fail", "Pass"])
ax.set_yticks([0,1], labels=["Fail", "Pass"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()

## 9. Feature analysis

In [ ]:
corr = df.select_dtypes(exclude=["object"]).corr()["G3"].drop("G3").sort_values(key=lambda s: s.abs(), ascending=False).head(10)
corr.sort_values().plot(kind="barh", figsize=(8,5))
plt.title("Top Numeric Correlations with Final Grade (G3)")
plt.xlabel("Correlation")
plt.tight_layout()
plt.show()

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=42, scoring="f1")
importance_df = pd.DataFrame({"Feature": X.columns, "Importance": perm.importances_mean}).sort_values("Importance", ascending=False).head(12)
importance_df

## 10. Additional experiment without G1 and G2

Because `G1` and `G2` are previous grades and are strongly related to `G3`, an additional experiment is performed without them. This helps discuss how model performance changes when direct grade-related predictors are removed.

In [ ]:
feature_cols_without_grades = [c for c in df.columns if c not in ["G1", "G2", "G3", "Pass_Fail"]]
X2 = df[feature_cols_without_grades]

cat2 = X2.select_dtypes(include=["object"]).columns.tolist()
num2 = X2.select_dtypes(exclude=["object"]).columns.tolist()
preprocess2 = ColumnTransformer([
    ("num", StandardScaler(), num2),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat2)
])

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.20, random_state=42, stratify=y)

quick_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, C=1, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=5, random_state=42, n_jobs=1),
    "SVM": SVC(C=1, kernel="linear", random_state=42)
}

secondary_results = []
for name, estimator in quick_models.items():
    pipe = Pipeline([("preprocess", preprocess2), ("clf", estimator)])
    pipe.fit(X2_train, y2_train)
    pred = pipe.predict(X2_test)
    secondary_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y2_test, pred), 4),
        "Precision": round(precision_score(y2_test, pred), 4),
        "Recall": round(recall_score(y2_test, pred), 4),
        "F1-score": round(f1_score(y2_test, pred), 4)
    })

secondary_df = pd.DataFrame(secondary_results).sort_values("F1-score", ascending=False).reset_index(drop=True)
secondary_df

## 11. Conclusion

The best model in the main experiment achieved strong performance because previous grades `G1` and `G2` are highly predictive of the final grade. When these columns were removed, the performance decreased, showing that earlier academic performance is the strongest predictor of final success. Other factors such as failures, study time, absences, and school support also help explain student performance.